# Concurrent Web Crawler

- Concurrent graph traversal problem over URLs with hostname filtering and fragment sanitization.
- The core behavior is discovering every reachable same-host page exactly once while crawling in parallel.
- Important invariants are sanitizing fragments before deduplication, preserving same-host filtering, and handling cycles safely.
- Similar patterns show up in crawlers, service discovery, dependency walkers, and distributed metadata collection.
- Focus on fragment variants, cross-host links, cycles, and pages that converge onto the same sanitized URL.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from threading import Event, Lock


class Solution:
    def crawl(self, startUrl: str, htmlParser: 'HtmlParser') -> list[str]:
        def sanitize(url: str) -> str:
            return url.split('#', 1)[0]

        def hostname(url: str) -> str:
            rest = url.split('://', 1)[1]
            return rest.split('/', 1)[0]

        start = sanitize(startUrl)
        host = hostname(start)
        visited = {start}
        lock = Lock()
        done = Event()
        pending = 1

        def worker(url: str) -> None:
            nonlocal pending
            try:
                for raw in htmlParser.getUrls(url):
                    nxt = sanitize(raw)
                    if hostname(nxt) != host:
                        continue
                    with lock:
                        if nxt in visited:
                            continue
                        visited.add(nxt)
                        pending += 1
                    executor.submit(worker, nxt)
            finally:
                with lock:
                    pending -= 1
                    if pending == 0:
                        done.set()

        with ThreadPoolExecutor(max_workers=8) as executor:
            executor.submit(worker, start)
            done.wait()

        return list(visited)


In [ ]:
def test(solution):
    class MockHtmlParser:
        def __init__(self, graph):
            self.graph = graph

        def getUrls(self, url):
            return self.graph.get(url, [])

    cases = [
        ((('http://example.com/page1', MockHtmlParser({
            'http://example.com/page1': ['http://example.com/page2', 'http://example.com/page3#sectionA'],
            'http://example.com/page2': ['http://example.net/page4#'],
            'http://example.com/page3': ['http://example.com/page1'],
        }))), ['http://example.com/page1', 'http://example.com/page2', 'http://example.com/page3']),
        ((('http://news.google.com/top', MockHtmlParser({
            'http://news.google.com/top': ['http://news.yahoo.com/home'],
            'http://news.yahoo.com/home': ['http://news.yahoo.com/news'],
        }))), ['http://news.google.com/top']),
        ((('http://site.com/a', MockHtmlParser({
            'http://site.com/a': ['http://site.com/b#frag1', 'http://site.com/b#frag2', 'http://site.com/c'],
            'http://site.com/b': ['http://site.com/d'],
            'http://site.com/c': ['http://other.com/x', 'http://site.com/d'],
            'http://site.com/d': ['http://site.com/e#'],
            'http://site.com/e': ['http://site.com/a'],
        }))), ['http://site.com/a', 'http://site.com/b', 'http://site.com/c', 'http://site.com/d', 'http://site.com/e']),
    ]
    for i, (args, expected) in enumerate(cases, 1):
        got = sorted(solution(*args))
        expected = sorted(expected)
        assert got == expected, f'case {i}: expected {expected}, got {got}'


In [ ]:
def current_solution(startUrl, htmlParser):
    return Solution().crawl(startUrl, htmlParser)

test(current_solution)
print('PASS')
